# Sources of Uncertainty in Strain Measurement

**A measurement you cannot bound is a number you cannot trust — so before we improve a strain reading, we account for everything that can push it off.**

Companion notebook to Chapter 46 — Sources of Uncertainty in Strain Measurement.

An *uncertainty budget* is a structured accounting of every source of error in a measurement system. Each source is expressed as a *standard uncertainty*, weighted by its *sensitivity coefficient* (the partial derivative of the output with respect to that input), and combined in quadrature. The budget serves two purposes: it produces the combined uncertainty of the final result, and it reveals which sources dominate — showing where effort to reduce uncertainty will actually pay off.

For a quarter-bridge strain measurement, the output is `ε = 4·V_out / (V_ex · GF)`. The sensitivity coefficients follow directly from the partial derivatives: gage factor `GF` and excitation voltage `V_ex` each appear in the denominator, so their uncertainty contributions scale with the current strain reading. A small percentage error in either produces a proportional strain error — which is why gage factor tolerance is often the largest *strain-proportional* contributor in a practical budget.

Contributions that do **not** depend on the strain level — thermal output (apparent strain from a self-temperature-compensation mismatch), installation-angle error, and residual transverse sensitivity — add fixed offsets regardless of load. At low strain these fixed-offset sources can easily dominate; at high strain the gage-factor and excitation terms take over. A complete budget must therefore be evaluated *at the operating point*, not in the abstract.

**This notebook demonstrates:**
1. A full uncertainty budget for a quarter-bridge strain measurement — sensitivity coefficients, individual contributions, and the combined standard uncertainty — evaluated at a nominal 1000 με.
2. A Pareto view of the budget that makes the dominant terms obvious at a glance.

In [ ]:
# In Google Colab these are pre-installed; uncomment to install locally.
# !pip install numpy pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This notebook is fully deterministic: the budget below is built entirely
# from fixed, published values, so there is no randomness to seed.

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

---
## 1 — Uncertainty budget for a quarter-bridge strain measurement

The budget below evaluates six sources at a nominal strain of 1000 με. Each source carries a standard uncertainty `u(x_i)` and a sensitivity coefficient `c_i = ∂ε/∂x_i` — the partial derivative that tells us how much a unit change in that input perturbs the strain output. The contribution of each source to the combined uncertainty is the product `c_i · u(x_i)`.

Two of the six sources need their raw specification converted into a standard uncertainty before they enter the budget. The gage-factor tolerance is a ±0.5 % *expanded* uncertainty stated at *k* = 2, so following Eq. 46.1 in the chapter we divide by two: `u_GF = GF × 0.005 / 2`. At 1000 με this yields a **2.5 με** contribution — exactly the figure the chapter derives. The remaining sources are entered as post-correction *residual* standard uncertainties: an STC-matched gage still leaves a thermal-output residual, and even a corrected rosette leaves a residual transverse-sensitivity term.

The six contributions come out as follows. Thermal output and transverse sensitivity are the largest at **3.0 με** each, gage factor follows at **2.5 με**, and excitation-voltage stability adds **2.0 με**. Installation-angle error from a 1° misalignment contributes **1.0 με** — a reminder that mechanical execution matters as much as electrical precision. The 24-bit ADC noise contributes an essentially negligible **0.15 με**; a good converter is rarely the limiting factor in a modern strain system. These three leading terms — thermal output, transverse sensitivity, and gage factor — are precisely the gage-property and thermal sources that Table 46.1 flags as principal; the excitation and ADC terms belong to the instrument chain and are developed fully in Chapter 47, and are included here only to show a complete budget.

Combined in quadrature, the six sources yield **u_c ≈ 5.41 με** and an expanded uncertainty **U ≈ 10.82 με** (*k* = 2). The budget is directly actionable: because the three leading terms are comparable, no single fix moves the total much — reducing the total meaningfully means attacking thermal output, transverse correction, *and* gage-factor tolerance together. Improving the ADC would accomplish nothing measurable. And because the gage-factor and excitation terms scale with strain while the thermal, transverse, and installation terms do not, the ranking shifts with load: at several thousand microstrain, gage factor and excitation take over.

In [ ]:
# Uncertainty budget for a quarter-bridge strain measurement.
#
# The strain equation is  ε = 4 · V_out / (V_ex · GF).  For each input we take
# its standard uncertainty u(x_i), multiply by the sensitivity coefficient
# c_i = ∂ε/∂x_i, and combine the contributions in quadrature to get u_c.

# --- Operating point and nominal bridge parameters ---
GAGE_FACTOR     = 2.07      # manufacturer lot-mean gage factor (dimensionless)
EXCITATION_V    = 5.0       # bridge excitation voltage, V
NOMINAL_STRAIN  = 1000e-6   # operating-point strain, dimensionless (= 1000 με)
MICROSTRAIN     = 1e6       # dimensionless strain -> microstrain (με)

# --- Type B specifications and how each converts to a standard uncertainty ---
GF_TOLERANCE_FRAC          = 0.005    # gage-factor tolerance, ±0.5% (Eq. 46.1)
GF_COVERAGE_K              = 2        # tolerance is an expanded uncertainty at k = 2
EXCITATION_STAB_V          = 0.010    # excitation stability, 10 mV RMS (1σ)
ADC_NOISE_V                = 0.4e-6   # bridge-output noise, ~0.4 µV RMS (24-bit DAQ)
STC_RESIDUAL_STRAIN        = 3e-6     # residual thermal output of an STC-matched gage
INSTALL_ANGLE_STRAIN       = 1e-6     # apparent strain from a 1° installation misalignment
TRANSVERSE_RESIDUAL_STRAIN = 3e-6     # residual after Kt correction (~0.3% of reading)


def quarter_bridge_sensitivities(gage_factor, excitation_v, strain):
    """Sensitivity coefficients c_i = ∂ε/∂x_i for the quarter-bridge readout.

    From ε = 4·V_out / (V_ex · GF), the partial derivatives are
        ∂ε/∂GF    = -ε / GF          (per unit gage factor)
        ∂ε/∂V_ex  = -ε / V_ex        (per volt)
        ∂ε/∂V_out =  4 / (V_ex · GF) (per volt)
    evaluated at the given operating-point `strain`.
    """
    return {
        'gage_factor': -strain / gage_factor,
        'excitation':  -strain / excitation_v,
        'output_volt':  4.0 / (excitation_v * gage_factor),
    }


def build_budget_table(budget):
    """Assemble the budget as a DataFrame of per-source contributions in με.

    `budget` maps a source name to (standard uncertainty u(x_i),
    sensitivity coefficient c_i, note). The contribution is |c_i · u(x_i)|,
    reported in microstrain.
    """
    rows = []
    for name, (u_xi, c_i, note) in budget.items():
        contribution_ue = abs(c_i * u_xi) * MICROSTRAIN
        rows.append({
            'Source': name,
            'u(x_i)': u_xi,
            'c_i': c_i,
            'contribution (με)': contribution_ue,
            'Note': note,
        })
    return pd.DataFrame(rows)


# Convert the gage-factor tolerance (±0.5%, k = 2) to a standard uncertainty.
u_gage_factor = GAGE_FACTOR * GF_TOLERANCE_FRAC / GF_COVERAGE_K  # Eq. 46.1

c = quarter_bridge_sensitivities(GAGE_FACTOR, EXCITATION_V, NOMINAL_STRAIN)

# Each entry: (standard uncertainty u(x_i), sensitivity coefficient c_i, note).
# The three fixed-offset sources are already expressed as an apparent strain,
# so their sensitivity coefficient is exactly 1.
budget = {
    'Gage factor GF':         (u_gage_factor,              c['gage_factor'], 'from +/-0.5% spec, k=2 (Eq. 46.1)'),
    'Excitation V_ex':        (EXCITATION_STAB_V,          c['excitation'],  '10 mV RMS stability'),
    'Output voltage V_out':   (ADC_NOISE_V,                c['output_volt'], '~0.4 uV ADC noise (24-bit)'),
    'STC thermal output':     (STC_RESIDUAL_STRAIN,        1.0,              'residual of STC-matched gage'),
    'Installation angle':     (INSTALL_ANGLE_STRAIN,       1.0,              '1 deg misalignment'),
    'Transverse sensitivity': (TRANSVERSE_RESIDUAL_STRAIN, 1.0,              'residual after Kt correction'),
}

budget_df = build_budget_table(budget)

# Combined standard uncertainty: root-sum-square of the individual contributions.
u_c_ue = np.sqrt(np.sum(budget_df['contribution (με)'] ** 2))
U_expanded_ue = GF_COVERAGE_K * u_c_ue  # expanded uncertainty at k = 2

print(budget_df.to_string(index=False, float_format=lambda v: f'{v:.4g}'))
print('-' * 78)
print(f'Combined standard uncertainty  u_c = {u_c_ue:6.2f} με')
print(f'Expanded uncertainty (k = 2)     U = {U_expanded_ue:6.2f} με')

---
## 2 — Which sources dominate? A Pareto view

A budget table tells you the numbers; a Pareto chart tells you the *story*. Sorting the contributions from largest to smallest makes the leading terms obvious and shows, at a glance, where reducing uncertainty would actually help. Because the individual terms combine in quadrature, the plot below also marks the combined standard uncertainty `u_c` — note that it is not the sum of the bars, and that it sits only a little above the largest single contributor.

In [ ]:
# Pareto chart of the individual uncertainty contributions (sorted, largest first).
pareto = budget_df.sort_values('contribution (με)', ascending=True)

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh(pareto['Source'], pareto['contribution (με)'], color='#3b6ea5')
ax.axvline(u_c_ue, color='#c1440e', linestyle='--', linewidth=1.5,
           label=f'Combined $u_c$ = {u_c_ue:.2f} με')

for source, value in zip(pareto['Source'], pareto['contribution (με)']):
    ax.text(value + 0.05, source, f'{value:.2f}', va='center', fontsize=9)

ax.set_xlabel('Contribution to combined uncertainty, $|c_i \\cdot u(x_i)|$ (με)')
ax.set_title('Quarter-bridge uncertainty budget at 1000 με (sorted)')
ax.set_xlim(0, u_c_ue * 1.15)
ax.legend(loc='lower right')
fig.tight_layout()
plt.show()

---
## Where to take this next

This budget is a starting point. A few concrete extensions:

1. **Sweep the operating point.** Wrap the budget in a loop over strain from 0 to 5000 με and plot each contribution versus strain. The gage-factor and excitation curves rise linearly while the thermal, transverse, and installation offsets stay flat — so you can read off the strain at which the dominant source changes. This makes the chapter's "evaluate at the operating point" argument concrete.
2. **Add the wiring and self-heating terms.** Fold in the lead-wire apparent strain of Eq. 46.7 and a self-heating contribution derived from the power density of Eq. 46.9, each as a Type B rectangular source (divide the bound by √3). Compare a two-wire versus a three-wire installation and watch the lead-wire term collapse.
3. **Cross-check the quadrature result with Monte Carlo.** Draw each input from its distribution (normal for the k = 2 gage-factor term, uniform for the rectangular bounds), propagate through `ε = 4·V_out/(V_ex·GF)`, and compare the sampled standard deviation to `u_c`. Seed the generator with `np.random.default_rng(42)` so the result is reproducible. Agreement confirms the linear GUM approximation; disagreement flags a source large enough that its nonlinearity matters.